# Saffron detection — Colab GPU workflow

Run this notebook on a T4 GPU. It is resumable: completed folds are skipped and the final cell zips the run outputs.

Runtime estimate: not yet measured on a completed T4 run. Run the smoke test first and do not treat smoke-test metrics as results.


In [ ]:
import os, subprocess, sys, json

print('GPU check:')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No GPU detected. Use Colab T4 or a Linux GPU host. This workflow is not valid on CPU for full training.')
print(r.stdout.strip())

print('GPU available; install the pinned project dependencies in the next setup cell.')


In [ ]:
# Dependency installation runs after cloning, in the repository root.
print('Continue to the repository and data setup cell.')


In [ ]:
# Get the repo and copy the private dataset archives from Google Drive
!git clone https://github.com/ariktadas144/Saffron-Authentication-Detector.git /content/saffron
%cd /content/saffron

# Put both original dataset ZIPs in Drive/MyDrive/saffron_data.
# If they are not there, this cell checks the Drive root and then offers browser upload.
from pathlib import Path
from google.colab import drive
from google.colab import files
import shutil

drive.mount('/content/drive')
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/saffron_data')
archive_files = sorted(DRIVE_DATA_DIR.glob('*.zip')) if DRIVE_DATA_DIR.is_dir() else []
if len(archive_files) != 2:
    drive_root_archives = sorted(Path('/content/drive/MyDrive').glob('*.zip'))
    if len(drive_root_archives) == 2:
        archive_files = drive_root_archives
        print('Found exactly 2 dataset archives in the My Drive root.')
if len(archive_files) != 2:
    print(
        f'Found {len(archive_files)} ZIP file(s) in {DRIVE_DATA_DIR}. '
        'Select the two original dataset ZIP files in the upload dialog. '
        'Alternatively, stop this cell, place them in Google Drive/My Drive/saffron_data, '
        'and rerun it. The GitHub repository does not contain private dataset files.'
    )
    uploaded = files.upload()
    archive_files = sorted(Path.cwd() / name for name in uploaded if name.lower().endswith('.zip'))
if len(archive_files) != 2:
    raise RuntimeError(
        f'Expected exactly 2 original dataset ZIP files, found {len(archive_files)}: '
        f'{[p.name for p in archive_files]}. Upload both original archives or put them in '
        'Google Drive/My Drive/saffron_data, then rerun this cell.'
    )
Path('data').mkdir(exist_ok=True)
for archive in archive_files:
    destination = Path('data') / archive.name
    shutil.copy2(archive, destination)
    print(f'Copied {archive.name} ({destination.stat().st_size:,} bytes)')

# Install the environment versions recorded in the project walkthrough.
install_command = [
    sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
    'numpy==2.4.2', 'pandas==2.2.3', 'scikit-learn==1.6.1',
    'scipy==1.15.2', 'Pillow==12.2.0', 'tensorflow==2.20.0', 'keras==3.13.0',
]
subprocess.run(install_command, check=True)
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('TensorFlow GPUs', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('TensorFlow did not detect a GPU after installing dependencies.')

print('Repo ready at', os.getcwd())
print('Dataset archives copied:', len(archive_files))


In [ ]:
# Prepare dataset, then verify that the manifest points to available images
import os, glob, subprocess, sys
import pandas as pd

if not os.path.exists('data/manifest.csv'):
    zips = sorted(glob.glob('data/*.zip'))
    if len(zips) != 2:
        raise RuntimeError(
            f'Expected 2 dataset archives in data/ but found {len(zips)}: {zips}. '
            'Run the preceding Drive-copy cell and place both original ZIP files in '
            'Google Drive/My Drive/saffron_data.'
        )
    command = [sys.executable, 'scripts/prepare_dataset.py', '--src', *zips, '--data', 'data', '--folds', '5']
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Using existing data/manifest.csv')

manifest = pd.read_csv('data/manifest.csv')
if manifest.empty or not {'path', 'label', 'group', 'fold'}.issubset(manifest.columns):
    raise RuntimeError('data/manifest.csv is empty or missing required columns: path,label,group,fold')
manifest_paths = manifest.path.astype(str).str.replace('\\', os.sep, regex=False)
missing_images = [path for path in manifest_paths if not os.path.isfile(path)]
if missing_images:
    raise FileNotFoundError(
        f'{len(missing_images)} manifest images are missing; first paths: {missing_images[:5]}. '
        'Re-run dataset preparation from the two original ZIP files in Drive.'
    )
print(f'Manifest ready: {len(manifest)} images, {manifest.group.nunique()} sample groups, {manifest.fold.nunique()} folds')

# run_cv.py needs the EXIF/capture table; this also creates the shortcut baselines.
subprocess.run([
    sys.executable, 'scripts/audit_shortcuts.py', '--manifest', 'data/manifest.csv',
    '--audit-out', 'data/audit_exif.csv',
    '--summary-out', 'runs/audit/shortcut_baselines.csv',
], check=True)


In [ ]:
# Fold 0 — default settings, run and resume-safe
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_repro --folds 0 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Label shuffle sanity check, fold 0
!python scripts/label_shuffle_test.py --manifest data/manifest.csv --out runs/audit/label_shuffle --real-out runs/cv_repro --fold 0 --seeds 101 202 303 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Folds 1–4 (resume-safe; completed folds are skipped if the cache files already exist)
!python scripts/run_cv.py --manifest data/manifest.csv --out runs/cv_repro --folds 1 2 3 4 --img-size 256 --batch 16 --epochs-warm 3 --epochs-ft 25 --mc-samples 16 --seed 42


In [ ]:
# Final aggregation and collection
!python scripts/collect_results.py --cv-dir runs/cv_repro --shuffle-dir runs/audit/label_shuffle --out runs/audit/collected_phase2_results.csv
!python scripts/run_cv.py --out runs/cv_repro --aggregate-only


In [ ]:
# Zip final results for download
import os, shutil, zipfile
archive = 'runs_for_download.zip'
if os.path.exists(archive):
    os.remove(archive)
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('runs'):
        for name in files:
            full = os.path.join(root, name)
            arcname = os.path.relpath(full, '.')
            zf.write(full, arcname)
print('Created', archive)
